# Distributed PEFT Fine-Tuning (Async Gradient Exchange)

3-node distributed LoRA fine-tuning of Phi-2 with FedAvg aggregation.

**Each node runs this notebook** with its own `CONFIG` settings.

| Node | Layers | Role |
|------|--------|------|
| A | 0-10 | Input parsing, instruction format recognition |
| B | 11-21 | Reasoning, content planning |
| C | 22-31 + head | Output generation, token selection |

## Section 0 — Install & Imports

In [ ]:
!pip install -q peft transformers datasets accelerate huggingface-hub safetensors requests matplotlib

In [ ]:
!git clone https://github.com/madch3m/peft-distributed.git /content/peft-distributed
import sys
sys.path.insert(0, "/content/peft-distributed")

In [ ]:
import os
import json
import time
import copy
from collections import defaultdict

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download, upload_file, upload_folder
from safetensors.torch import load_file, save_file

print(f"torch {torch.__version__}  CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Section 1 — CONFIG

**Set these per node before running.**

In [ ]:
CONFIG = {
    # === Identity ===
    "node_id": "node_a",          # "node_a", "node_b", or "node_c"
    "layer_start": 0,              # Node A: 0, Node B: 11, Node C: 22
    "layer_end": 10,               # Node A: 10, Node B: 21, Node C: 31

    # === HuggingFace Hub ===
    "hf_token": "hf_YOUR_TOKEN",  # Write-access token
    "repo_id": "your-org/your-model-repo",
    # Set False when testing against a repo that already exists (skips create_repo in Hub init).
    "hub_create_repo": True,

    # === Aggregator ===
    "aggregator_url": "https://your-username-peft-aggregator.hf.space",
    "node_secret": "choose_any_shared_secret_string",
    # Optional: if the Space sets STATUS_READ_SECRET, put the same value here (for polling /status).
    "status_read_secret": "",

    # === LoRA ===
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj", "k_proj", "dense", "fc1", "fc2"],

    # === Training ===
    "base_model": "microsoft/phi-2",
    "max_seq_len": 512,
    "batch_size": 2,
    "grad_accum_steps": 4,
    "num_rounds": 3,
    "steps_per_round": 100,
    "learning_rate": 2e-4,

    # === Async gradient exchange ===
    "grad_sync_every_n_steps": 10,
    "remote_grad_lr": 0.5,
    "grad_staleness_threshold": 3,
}

# Optional: domain-specific datasets per node
# DOMAIN_MAP = {
#     "node_a": {"hf_path": "medalpaca/medical_meadow_alpaca", "domain": "medical"},
#     "node_b": {"hf_path": "nguha/legalbench",                "domain": "legal"},
#     "node_c": {"hf_path": "sahil2801/CodeAlpaca-20k",        "domain": "code"},
# }
# CONFIG["domain"]  = DOMAIN_MAP[CONFIG["node_id"]]["domain"]
# CONFIG["hf_path"] = DOMAIN_MAP[CONFIG["node_id"]]["hf_path"]

os.environ["HF_TOKEN"] = CONFIG["hf_token"]
api = HfApi(token=CONFIG["hf_token"])

print(f"Node: {CONFIG['node_id']}  Layers: {CONFIG['layer_start']}-{CONFIG['layer_end']}")

## Section 2 — Hub Init (Node A only)

Run this cell **once** from Node A to create `training_state.json` on the Hub.

In [ ]:
if CONFIG["node_id"] == "node_a":
    if CONFIG.get("hub_create_repo", True):
        try:
            api.create_repo(CONFIG["repo_id"], exist_ok=True, private=True)
            print(f"Repo ready: {CONFIG['repo_id']}")
        except Exception as e:
            print(f"Repo creation: {e}")
    else:
        print(
            "Skipping create_repo (hub_create_repo=False); "
            f"using existing repo {CONFIG['repo_id']!r}."
        )

    # Initialize training state
    training_state = {
        "current_round": 1,
        "last_merged_round": 0,
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    state_path = "/tmp/training_state.json"
    with open(state_path, "w") as f:
        json.dump(training_state, f, indent=2)

    api.upload_file(
        path_or_fileobj=state_path,
        path_in_repo="training_state.json",
        repo_id=CONFIG["repo_id"],
        commit_message="Init training state",
    )
    print("training_state.json uploaded.")
else:
    print(f"Skipping Hub init — this is {CONFIG['node_id']}, not node_a.")

## Section 3 — Dataset

Load and tokenize Alpaca 2K (or domain-specific dataset).

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load dataset — default: Alpaca
hf_path = CONFIG.get("hf_path", "tatsu-lab/alpaca")
raw = load_dataset(hf_path, split="train")

# Normalise to {text} format
def normalise(example):
    instruction = (
        example.get("instruction")
        or example.get("question")
        or example.get("input", "")
    )
    response = (
        example.get("output")
        or example.get("answer")
        or example.get("response", "")
    )
    return {"text": f"### Instruction:\n{instruction}\n\n### Response:\n{response}"}

dataset = raw.map(normalise).select(range(min(2000, len(raw))))

# Tokenize
def tokenize(example):
    out = tokenizer(
        example["text"],
        truncation=True,
        max_length=CONFIG["max_seq_len"],
        padding="max_length",
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)
tokenized.set_format("torch")

dataloader = DataLoader(tokenized, batch_size=CONFIG["batch_size"], shuffle=True)
print(f"Dataset: {len(tokenized)} samples, {len(dataloader)} batches")

## Section 4 — Model + LoRA

Load Phi-2, attach LoRA adapters, freeze base weights.

In [ ]:
# Load base model — newer transformers expects config.pad_token_id; Phi-2 config may omit it
model_config = AutoConfig.from_pretrained(
    CONFIG["base_model"],
    trust_remote_code=True,
)
_pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
if getattr(model_config, "pad_token_id", None) is None:
    try:
        model_config.pad_token_id = _pad
    except AttributeError:
        object.__setattr__(model_config, "pad_token_id", _pad)

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    config=model_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Attach LoRA
lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Freeze/unfreeze based on layer partition
owned_params = []
for name, param in model.named_parameters():
    if "lora_" in name:
        # Extract layer number from parameter name
        layer_num = None
        for part in name.split("."):
            if part.isdigit():
                layer_num = int(part)
                break

        if layer_num is not None and CONFIG["layer_start"] <= layer_num <= CONFIG["layer_end"]:
            param.requires_grad = True
            owned_params.append((name, param))
        else:
            param.requires_grad = False

print(f"\nOwned adapter params: {len(owned_params)}")
total_owned = sum(p.numel() for _, p in owned_params)
print(f"Owned trainable parameters: {total_owned:,}")

## Section 5 — Gradient Hooks

Register hooks on owned adapter parameters to capture gradients for async exchange.

In [ ]:
grad_buffer = {}
hook_handles = []

def register_hooks():
    """Register backward hooks on owned LoRA params to accumulate gradients."""
    global hook_handles
    # Clear existing hooks
    for h in hook_handles:
        h.remove()
    hook_handles = []
    grad_buffer.clear()

    for name, param in owned_params:
        grad_buffer[name] = torch.zeros_like(param.data)

        def make_hook(param_name):
            def hook_fn(grad):
                grad_buffer[param_name] += grad.detach().clone()
            return hook_fn

        handle = param.register_hook(make_hook(name))
        hook_handles.append(handle)

    print(f"Registered {len(hook_handles)} gradient hooks.")

register_hooks()

## Section 6 — Optimizer

AdamW on owned params only.

In [ ]:
optimizer = torch.optim.AdamW(
    [p for _, p in owned_params],
    lr=CONFIG["learning_rate"],
    weight_decay=0.01,
)

print(f"Optimizer: AdamW, lr={CONFIG['learning_rate']}, params={len(owned_params)}")

## Section 7 — Training Loop

Forward, backward, gradient accumulation, async push/pull of gradients.

In [ ]:
def push_gradients(step_num):
    """Push accumulated gradients to Hub for other nodes."""
    grad_state = {k: v.half() for k, v in grad_buffer.items()}
    save_path = f"/tmp/{CONFIG['node_id']}_grads.safetensors"
    save_file(grad_state, save_path)

    meta = {"node_id": CONFIG["node_id"], "step": step_num, "round": current_round}
    meta_path = f"/tmp/{CONFIG['node_id']}_grad_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f)

    api.upload_file(
        path_or_fileobj=save_path,
        path_in_repo=f"gradients/{CONFIG['node_id']}/grads.safetensors",
        repo_id=CONFIG["repo_id"],
        commit_message=f"{CONFIG['node_id']} grads step {step_num}",
    )
    api.upload_file(
        path_or_fileobj=meta_path,
        path_in_repo=f"gradients/{CONFIG['node_id']}/meta.json",
        repo_id=CONFIG["repo_id"],
        commit_message=f"{CONFIG['node_id']} grad meta step {step_num}",
    )

    # Reset buffer
    for k in grad_buffer:
        grad_buffer[k].zero_()


def pull_and_apply_remote_gradients(local_step):
    """Pull gradients from other nodes, apply with staleness check."""
    other_nodes = [n for n in ["node_a", "node_b", "node_c"] if n != CONFIG["node_id"]]

    for other in other_nodes:
        try:
            meta_path = hf_hub_download(
                repo_id=CONFIG["repo_id"],
                filename=f"gradients/{other}/meta.json",
                token=CONFIG["hf_token"],
            )
            with open(meta_path) as f:
                meta = json.load(f)

            staleness = local_step - meta["step"]
            if staleness >= CONFIG["grad_staleness_threshold"]:
                print(f"  Skipping {other} grads (staleness={staleness})")
                continue

            grad_path = hf_hub_download(
                repo_id=CONFIG["repo_id"],
                filename=f"gradients/{other}/grads.safetensors",
                token=CONFIG["hf_token"],
            )
            remote_grads = load_file(grad_path)

            applied = 0
            for name, param in owned_params:
                if name in remote_grads:
                    param.data -= CONFIG["remote_grad_lr"] * remote_grads[name].to(param.device).float()
                    applied += 1

            print(f"  Applied {applied} remote grads from {other} (staleness={staleness})")

        except Exception as e:
            print(f"  Could not pull from {other}: {e}")


# --- Main training loop ---
loss_history = []
staleness_log = []

for current_round in range(1, CONFIG["num_rounds"] + 1):
    print(f"\n{'='*60}")
    print(f"ROUND {current_round}/{CONFIG['num_rounds']}")
    print(f"{'='*60}")

    model.train()
    data_iter = iter(dataloader)
    round_losses = []

    # Snapshot adapter state at round start (for delta computation)
    adapter_start = {name: param.data.clone() for name, param in owned_params}

    for step in range(1, CONFIG["steps_per_round"] + 1):
        # Get batch (cycle dataloader)
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            batch = next(data_iter)

        batch = {k: v.to(model.device) for k, v in batch.items()}

        # Forward + backward
        outputs = model(**batch)
        loss = outputs.loss / CONFIG["grad_accum_steps"]
        loss.backward()

        # Gradient accumulation step
        if step % CONFIG["grad_accum_steps"] == 0:
            optimizer.step()
            optimizer.zero_grad()

        round_losses.append(loss.item() * CONFIG["grad_accum_steps"])

        # Async gradient exchange
        if step % CONFIG["grad_sync_every_n_steps"] == 0:
            global_step = (current_round - 1) * CONFIG["steps_per_round"] + step
            print(f"  Step {step}: pushing grads...")
            push_gradients(global_step)
            print(f"  Step {step}: pulling remote grads...")
            pull_and_apply_remote_gradients(global_step)

        if step % 10 == 0:
            avg_loss = np.mean(round_losses[-10:])
            print(f"  Step {step}/{CONFIG['steps_per_round']}  loss={avg_loss:.4f}")

    loss_history.extend(round_losses)
    print(f"Round {current_round} avg loss: {np.mean(round_losses):.4f}")

## Section 8 — Round-End Sync

Push adapter state, notify aggregator, wait for FedAvg.

In [ ]:
from aggregator_client import AggregatorMergeFailed, notify_aggregator, poll_for_next_round


def round_end_sync(round_num):
    """Push adapter state, notify aggregator, poll for next round."""
    # Save full adapter state
    adapter_save_dir = f"/tmp/adapter_{CONFIG['node_id']}"
    model.save_pretrained(adapter_save_dir)

    # Upload to Hub under node folder
    api.upload_folder(
        folder_path=adapter_save_dir,
        path_in_repo=CONFIG["node_id"],
        repo_id=CONFIG["repo_id"],
        commit_message=f"{CONFIG['node_id']} adapter state round {round_num}",
    )
    print(f"Adapter state uploaded for round {round_num}.")

    # Notify aggregator (raises AggregatorMergeFailed if FedAvg/Hub merge fails)
    try:
        result = notify_aggregator(
            aggregator_url=CONFIG["aggregator_url"],
            node_id=CONFIG["node_id"],
            secret_key=CONFIG["node_secret"],
            round_num=round_num,
        )
    except AggregatorMergeFailed as e:
        print("Aggregator merge_failed:", e.payload.get("merge_result", e))
        raise
    print(f"Aggregator response: {result}")

    # Wait for all nodes + FedAvg
    if result.get("status") != "round_complete":
        print("Waiting for other nodes...")
        status = poll_for_next_round(
            aggregator_url=CONFIG["aggregator_url"],
            current_round=round_num,
            status_secret=CONFIG.get("status_read_secret") or None,
        )
        print(f"Next round ready: {status}")

    # Load merged adapter from Hub
    try:
        merged_path = hf_hub_download(
            repo_id=CONFIG["repo_id"],
            filename=f"merged/round_{round_num}/adapter_model.safetensors",
            token=CONFIG["hf_token"],
        )
        merged_state = load_file(merged_path)
        adapter_state = dict(model.named_parameters())
        loaded = 0
        for key, tensor in merged_state.items():
            if key in adapter_state:
                adapter_state[key].data.copy_(tensor.to(adapter_state[key].device))
                loaded += 1
        print(f"Loaded {loaded} merged adapter params from FedAvg.")
    except Exception as e:
        print(f"Could not load merged adapter: {e}")


# Run sync for the last completed round
round_end_sync(current_round)

## Section 9 — Evaluation

Instruction-following accuracy on held-out prompts.

In [ ]:
eval_prompts = [
    "### Instruction:\nExplain photosynthesis in three sentences.\n\n### Response:\n",
    "### Instruction:\nList 5 benefits of regular exercise.\n\n### Response:\n",
    "### Instruction:\nWrite a Python function to check if a number is prime.\n\n### Response:\n",
    "### Instruction:\nWhat is the capital of France? Answer in one word.\n\n### Response:\n",
    "### Instruction:\nSummarize the theory of relativity for a 10-year-old.\n\n### Response:\n",
]

model.eval()
print("\n" + "="*60)
print("EVALUATION")
print("="*60)

for prompt in eval_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"\n{'─'*40}")
    print(response)
    print()

## Section 10 — Plots

Loss curves and round history.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(loss_history, alpha=0.3, label="per-step")
window = 10
if len(loss_history) >= window:
    smoothed = np.convolve(loss_history, np.ones(window)/window, mode="valid")
    axes[0].plot(range(window-1, len(loss_history)), smoothed, label=f"{window}-step avg", linewidth=2)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title(f"Training Loss — {CONFIG['node_id']}")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Round boundaries
steps_per_round = CONFIG["steps_per_round"]
for r in range(1, CONFIG["num_rounds"] + 1):
    axes[0].axvline(x=r * steps_per_round, color="red", linestyle="--", alpha=0.5)

# Per-round average loss
round_avgs = []
for r in range(CONFIG["num_rounds"]):
    start = r * steps_per_round
    end = start + steps_per_round
    if end <= len(loss_history):
        round_avgs.append(np.mean(loss_history[start:end]))

axes[1].bar(range(1, len(round_avgs) + 1), round_avgs, color="steelblue")
axes[1].set_xlabel("Round")
axes[1].set_ylabel("Avg Loss")
axes[1].set_title(f"Per-Round Average Loss — {CONFIG['node_id']}")
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("/tmp/training_plots.png", dpi=150)
plt.show()
print("Plots saved to /tmp/training_plots.png")